# 📘 코드 해설판 (Python 초보자용)

이 노트북은 원본 `3. Task_2.ipynb` 에 **번호(`# 1`, `# 2` …)가 붙은 코드 셀마다 한국어 해설**을 덧붙인 파일입니다.
코드 셀 바로 **위**의 `💡 [# N] 코드 설명` 칸이 해설입니다. 원본 코드와 원본 설명은 그대로입니다.

**이 노트북의 목표:** 웹 검색만으로는 답하기 어려운 **금융 질문**(주가, 배당, EPS 등)을 **지식그래프(KG) API**에서 정확한 데이터를 가져와 답하게 만듭니다.

```
질문 ──▶ ① Query Generator : LLM이 질문에서 "회사명 / 지표 / 날짜"를 뽑아 JSON으로 정리
              │
              ▼
         ② Query Executor  : 그 JSON에 맞는 KG API를 호출해 데이터 가져오기
              │
              ▼
         ③ Reader          : 가져온 데이터 + 질문 → LLM → 답
```

- # 1 ~ # 5 : 준비, KG API 호출 클래스(`CRAG`)
- # 6 ~ # 9 : Query Generator (질문 → 구조화된 JSON)
- # 10 ~ # 13 : Query Executor와 `KGQueryEngine`
- # 14 ~ # 17 : Reader, 그리고 LLM + KG RAG
- # 18 ~ # 20 : LLM + 웹 검색 + KG를 모두 합친 RAG

**용어:** *티커(ticker)* = 주식 종목 코드(예: Microsoft → MSFT), *EPS* = 주당순이익, *P/E ratio* = 주가수익비율, *marketCap* = 시가총액, *dividend* = 배당

#📓 TASK #2: KNOWLEDGE GRAPH AND WEB AUGMENTATION

This task introduces mock APIs to access information from underlying mock Knowledge Graphs (KGs), with structured data possibly related to the questions. Participants use mock APIs, inputting parameters derived from the questions, to retrieve relevant data for answer formulation. The evaluation focuses on the systems' ability to query structured data and integrate information from various sources into comprehensive answers.

<img src="https://i.ibb.co/cXqvBZq/2024-12-07-3-09-46.png">

### Steps in RAG with Mock API
1. The model receives an input query to which a response is required.
2. Retriever retrieves relevant chunks from the recieved web pages that are pertinent to the input query.
3. Mock API retrieves relevant information from the Mock KG that are pertinent to the input query.
4. The large language model then generates a response, informed by both the original query and the retrieved information.

This practice class will be comprised of five sections.  
  
### I. Implementing a Mock KG Query Engine
### II. Implementing a Reader
### III. Implementing a LLM + Mock KG
### IV. Implementing a LLM + Web Search Results +Mock KG

## I. Implementing a Mock KG Query Engine

As you observed in the previous session, we can send specific queries to the Mock API connected to the Knowledge Graph (KG). The results obtained from the KG through the Mock API can then be utilized in the LLM’s inference stage.

Ultimately, what we will implement in this session is a connection between the existing RAG and the KG. Specifically, we will build the Mock KG query engine that serves as this connection.

More specifically, we will create a Mock KG query engine that generates a query from an input question **belonging to the finance domain**, sends it to the KG, and retrieves the relevant information.

The process will be carried out in the following five steps.

1. Preparing Python Packages
2. Preparing Mock APIs
3. Implementing a Query Generator
4. Implementing a Query Executor
5. Implementing a Mock KG Query Engine

### 1. Preparing Python Packages

As always, we will install and import the necessary python packages for use.  

The important point is that the **external IP address** of the KG we will use must be set correctly. If the configuration is incorrect, errors may occur in subsequent code execution.

### 💡 [# 1] 코드 설명 — 라이브러리 설치

**한 줄 요약:** 실습에 필요한 라이브러리를 설치합니다.

- `os.environ.pop("PYTHONPATH", None)` : 라이브러리 충돌 방지
- `!pip install ...` : 터미널 설치 명령(줄 끝 `\` 는 줄 이어 쓰기)
- 실제로 쓰는 주요 라이브러리: `openai`, `llama-index`, `beautifulsoup4`, `blingfire`, 그리고 함께 설치되는 `requests`(서버 통신)

In [ ]:
# 1
### YOUR CODE HERE ###
import os
os.environ.pop("PYTHONPATH", None)

!pip install llama-index llama-index-readers-wikipedia wikipedia \
    llama-index-llms-openai llama-index-embeddings-huggingface packaging>=24.0 \
    trulens trulens-providers-openai openai langchain nltk>=3.8.1 \
    streamlit==1.35.0 watchdog kubernetes==26.1.0 blingfire beautifulsoup4 \
    sentence-transformers ray textwrap3 scikit-learn transformers --quiet

### 💡 [# 2] 코드 설명 — 도구 불러오기와 환경 설정

**한 줄 요약:** 필요한 도구를 불러오고, OpenAI 키와 KG 서버 주소를 환경변수에 등록합니다.

- `import requests` : 서버에 요청을 보내는 도구
- `import numpy as np`, `bz2`, `json`, `BeautifulSoup` 등 : 앞 노트북과 동일한 도구들
- `os.environ["OPENAI_API_KEY"] = "sk-..."` : ⚠️ 본인 키로 교체
- `os.environ["CRAG_SERVER"] = " http://10.2.0.165:8000"` : KG API 서버 주소 저장 (강의실 내부망 주소)
- `nltk.download('punkt')` : 문장 분리용 데이터 다운로드

> 🔰 서버 주소 문자열 맨 앞에 **공백 한 칸**이 들어가 있습니다. 대부분 문제없이 동작하지만, 접속 오류가 나면 이 공백을 지워 보세요.

In [ ]:
# 2
### YOUR CODE HERE ###

from typing import List
import requests
import numpy as np
import bz2
import json
from blingfire import text_to_sentences_and_offsets
from collections import defaultdict
from typing import Any, Dict, List
from bs4 import BeautifulSoup
import os
import openai

os.environ["OPENAI_API_KEY"] = "sk-..." #copy your api key
os.environ["CRAG_SERVER"] = " http://10.2.0.165:8000"

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, get_response_synthesizer
from llama_index.readers.wikipedia import WikipediaReader
from llama_index.core.node_parser import SentenceSplitter

import textwrap

import nltk
nltk.download('punkt')

### 💡 [# 3] 코드 설명 — 설정값(상수)

**한 줄 요약:** 조각 길이, 참고자료 최대 길이 같은 설정 숫자를 이름 붙여 저장합니다.

- `MAX_CONTEXT_REFERENCES_LENGTH = 4000` : LLM에 넘길 참고자료 최대 글자 수 (# 14의 Reader에서 사용)
- 나머지는 앞 노트북과 같은 값입니다. 대문자 이름은 "상수"라는 관례입니다.

In [ ]:
# 3
### YOUR CODE HERE ###

# Define the number of context sentences to consider for generating an answer.
NUM_CONTEXT_SENTENCES = 20
# Set the maximum length for each context sentence (in characters).
MAX_CONTEXT_SENTENCE_LENGTH = 1000
# Set the maximum context references length (in characters).
MAX_CONTEXT_REFERENCES_LENGTH = 4000
# Sentence Transformer Parameters
SENTENTENCE_TRANSFORMER_BATCH_SIZE = 128 # TUNE THIS VARIABLE depending on the size of your embedding model and GPU mem available

### 2. Preparing Mock APIs

We have previously seen an example of an API that sends a query to the Mock KG and retrieves related information. Here, we will explain in more detail what types of APIs may exist.

The example below demonstrates a Mock API used in the KDD Cup. Each method is designed to connect to the KG (represented as self.server) and send the required request using the requests library. Simultaneously, it receives the results and finally returns the data in **JSON format**.

Here, the `requests` library is a Python package that simplifies making HTTP requests. Using methods like `GET `or `POST` from the requests library allows us to send HTTP requests, but it requires some understanding of computer networks, so we will skip the detailed explanation. If you want to learn more, I recommend researching it independently.

In any case, this is the connection bridge we have been discussing with the Mock KG. Please review the code below and check which methods are available.




### 💡 [# 4] 코드 설명 — KG API를 호출하는 클래스 `CRAG`

**한 줄 요약:** 금융 지식그래프 서버의 기능들을 파이썬 함수처럼 부를 수 있게 포장한 클래스입니다.

- `__init__` : `os.environ.get('CRAG_SERVER', "기본주소")` — 환경변수에 서버 주소가 있으면 그것을, 없으면 기본 주소를 `self.server` 에 저장
- 그 아래 함수 9개는 **모두 같은 모양**입니다. 하나만 이해하면 됩니다.

```python
def finance_get_eps(self, ticker_name:str):
    url = self.server + '/finance/get_eps'      # ① 서버주소 + 기능별 경로
    headers={'accept': "application/json"}      # ② "JSON으로 답해 주세요"
    data = {'query': ticker_name}               # ③ 보낼 내용
    result = requests.post(url, json=data, headers=headers)  # ④ POST 요청 전송
    return json.loads(result.text)              # ⑤ 응답 글자를 사전으로 변환
```

| 함수 | 하는 일 |
|---|---|
| `finance_get_company_name` | 검색어와 비슷한 **회사 정식 이름** 찾기 |
| `finance_get_ticker_by_name` | 회사 이름 → **티커** |
| `finance_get_price_history` | 일별 주가 기록 |
| `finance_get_detailed_price_history` | 더 촘촘한(분 단위) 주가 기록 |
| `finance_get_dividends_history` | 배당 기록 |
| `finance_get_market_capitalization` | 시가총액 |
| `finance_get_eps` | 주당순이익(EPS) |
| `finance_get_pe_ratio` | P/E 비율 |
| `finance_get_info` | 그 밖의 종합 정보 |

> 🔰 **GET** 은 "정보 주세요", **POST** 는 "이 내용을 보낼 테니 처리해 주세요"에 주로 쓰는 요청 방식입니다. 참고로 `__init__` 의 `server` 매개변수는 받기만 하고 쓰이지 않습니다.

In [ ]:
# 4
### YOUR CODE HERE ###

class CRAG(object):
    def __init__(self, server = None):
        self.server = os.environ.get('CRAG_SERVER', "http://10.2.0.165:8000")

    def finance_get_company_name(self, query:str):
        url = self.server + '/finance/get_company_name'
        headers={'accept': "application/json"}
        data = {'query': query}
        result = requests.post(url, json=data, headers=headers)
        return json.loads(result.text)

    def finance_get_ticker_by_name(self, query:str):
        url = self.server + '/finance/get_ticker_by_name'
        headers={'accept': "application/json"}
        data = {'query': query}
        result = requests.post(url, json=data, headers=headers)
        return json.loads(result.text)

    def finance_get_price_history(self, ticker_name:str):
        url = self.server + '/finance/get_price_history'
        headers={'accept': "application/json"}
        data = {'query': ticker_name}
        result = requests.post(url, json=data, headers=headers)
        return json.loads(result.text)

    def finance_get_detailed_price_history(self, ticker_name:str):
        url = self.server + '/finance/get_detailed_price_history'
        headers={'accept': "application/json"}
        data = {'query': ticker_name}
        result = requests.post(url, json=data, headers=headers)
        return json.loads(result.text)

    def finance_get_dividends_history(self, ticker_name:str):
        url = self.server + '/finance/get_dividends_history'
        headers={'accept': "application/json"}
        data = {'query': ticker_name}
        result = requests.post(url, json=data, headers=headers)
        return json.loads(result.text)

    def finance_get_market_capitalization(self, ticker_name:str):
        url = self.server + '/finance/get_market_capitalization'
        headers={'accept': "application/json"}
        data = {'query': ticker_name}
        result = requests.post(url, json=data, headers=headers)
        return json.loads(result.text)

    def finance_get_eps(self, ticker_name:str):
        url = self.server + '/finance/get_eps'
        headers={'accept': "application/json"}
        data = {'query': ticker_name}
        result = requests.post(url, json=data, headers=headers)
        return json.loads(result.text)

    def finance_get_pe_ratio(self, ticker_name:str):
        url = self.server + '/finance/get_pe_ratio'
        headers={'accept': "application/json"}
        data = {'query': ticker_name}
        result = requests.post(url, json=data, headers=headers)
        return json.loads(result.text)

    def finance_get_info(self, ticker_name:str):
        url = self.server + '/finance/get_info'
        headers={'accept': "application/json"}
        data = {'query': ticker_name}
        result = requests.post(url, json=data, headers=headers)
        return json.loads(result.text)

The code below is a simple example that allows you to test one of the APIs mentioned above.  

You can review the available methods and experiment freely as you wish.

### 💡 [# 5] 코드 설명 — API 직접 호출해 보기

**한 줄 요약:** "microsoft"를 예로 회사명 → 티커 → EPS 순서로 API를 호출해 결과를 확인합니다.

- `pretty_json_print(data)` : 데이터를 들여쓰기해 보기 좋게 출력하는 함수
- `api = CRAG()` : API 호출 객체 생성 (**이후 셀들이 이 `api` 를 계속 사용**)
- `metric = "eps"` : 알고 싶은 지표
- `api.finance_get_company_name("microsoft")` : 회사 이름 후보 목록
- `result["result"][0]` : 응답 사전의 `"result"` 항목(리스트)에서 **첫 번째** 후보
- `api.finance_get_ticker_by_name(...)` : 그 회사의 티커
- `if / elif / else` : **여러 갈래 조건문.** `metric` 값에 따라 해당하는 API 하나만 호출합니다. 어디에도 해당하지 않으면 `else` 의 `finance_get_info` 호출

> 🔰 `metric` 을 `'price'`, `'dividend'` 등으로 바꿔 실행해 보세요.

In [ ]:
# 5
### YOUR CODE HERE ###

def pretty_json_print(data):
    json_string = json.dumps(data, indent=4)
    lines = json_string.splitlines()
    formatted_lines = "\n\n".join(lines)
    print(formatted_lines)

api = CRAG()

metric = "eps"

result = api.finance_get_company_name("microsoft")
pretty_json_print(result)
ticker_name = api.finance_get_ticker_by_name(result["result"][0])
pretty_json_print(ticker_name)

if metric == 'price':
    response = api.finance_get_price_history(ticker_name['result'])
elif metric == 'dividend':
    response = api.finance_get_dividends_history(ticker_name['result'])
elif metric == 'p/e ratio':
    response = api.finance_get_pe_ratio(ticker_name['result'])
elif metric == 'eps':
    response = api.finance_get_eps(ticker_name['result'])
elif metric == 'marketcap' :
    response = api.finance_get_market_capitalization(ticker_name['result'])
else:
    response = api.finance_get_info(ticker_name['result'])

pretty_json_print(response)

### 3. Implementing a Query Generator

Creating the API, as shown above, is a good step. However, the important point is that the API requires a specific input format to function correctly.

Unfortunately, the CRAG dataset questions do not explicitly indicate which words or terms can be used as inputs for the API. To use the API effectively, we need to extract or generate the required terms from the question and provide them as inputs to the API.

There are various methods for this task. For example, in the field of **Named Entity Recognition (NER)**, techniques are studied to extract important words (entities) from a given sentence. Using a model developed in this field could be an excellent approach.

However, this would require loading additional models. Fortunately, we already have an LLM. Therefore, we can instruct the LLM to directly extract elements that can serve as inputs for the API.

Below is the prompt provided to the LLM for this purpose. It includes a breakdown of which entities are usable as API inputs for each domain. Reviewing this breakdown would also be a valuable exercise.

#### Designing prompts

### 💡 [# 6] 코드 설명 — 질문 분석용 프롬프트

**한 줄 요약:** LLM에게 "질문을 읽고 분야와 핵심 정보를 JSON으로 뽑아라"라고 시키는 지시문입니다.

지시 내용:
1. 질문의 **분야(domain)** 판별: finance / sports / music / movie / encyclopedia / other
2. finance 질문이면 다음 정보 추출
   - `market_identifier` : 회사 이름이나 종목 코드
   - `metric` : 묻고 있는 지표 (price, dividend, P/E ratio, EPS, marketCap, other 중 하나)
   - `datetime` : 묻는 시점 (없으면 질문 시각 사용)
3. 결과는 **평평한(FLAT) JSON** 으로, 설명 없이 JSON만 출력

> 🔰 예: "애플의 어제 종가는?" → `{"domain": "finance", "market_identifier": "Apple", "metric": "price", "datetime": "..."}`
> 이렇게 **자유로운 문장을 컴퓨터가 처리하기 쉬운 형태로 바꾸는 것**이 이 단계의 목적입니다.

In [ ]:
# 6
### YOUR CODE HERE ###

entity_extract_template = """
You are given a Query and Query Time. Do the following:

1) Determine the domain the query is about. The domain should be one of the following: "finance", "sports", "music", "movie", "encyclopedia". If none of the domain applies, use "other". Use "domain" as the key in the result json.

2) Extract structured information from the query. Include different keys into the result json depending on the domains, and put them DIRECTLY in the result json. Here are the rules:

For `finance` queries, these are possible keys:
- `market_identifier`: stock identifiers including individual company names, stock symbols.
- `metric`: financial metrics that the query is asking about. This must be one of the following: `price`, `dividend`, `P/E ratio`, `EPS`, `marketCap`, and `other`.
- `datetime`: time frame that query asks about. When datetime is not explicitly mentioned, use `Query Time` as default.


Return the results in a FLAT json.

*NEVER include ANY EXPLANATION or NOTE in the output, ONLY OUTPUT JSON*
"""

### 💡 [# 7] 코드 설명 — LLM 입력 조립 함수

**한 줄 요약:** 질문을 받아 (시스템 지시문 + 사용자 메시지) 형태로 만들어 줍니다.

- `user_message += f"Query: {query}\n"` : "Query: 질문" 형태의 글
- `{"role": "system", "content": entity_extract_template}` : # 6의 지시문
- `{"role": "user", "content": user_message}` : 질문
- `return llm_input`

In [ ]:
# 7
### YOUR CODE HERE ###

def prompt_generator(query):
    user_message = ""
    user_message += f"Query: {query}\n"

    llm_input = [
      {"role": "system", "content": entity_extract_template},
      {"role": "user", "content": user_message},
    ]

    return llm_input

#### Generating Queries

Now, let’s actually deliver this prompt to the LLM and generate the query that will be sent to the API.

### 💡 [# 8] 코드 설명 — 질문을 구조화된 JSON으로 바꾸기

**한 줄 요약:** LLM을 호출해 질문에서 정보를 뽑고, 금융 질문인지 여부도 함께 돌려주는 함수입니다.

**`generate_query(query)`**
- `prompt_generator(query)` → `oai_client.chat.completions.create(...)` : LLM 호출, 답변 글(JSON 모양의 **문자열**)을 `completion` 에 저장
- `try: completion = json.loads(completion)` : 문자열 → 사전 변환 시도
- `except: completion = extract_json_objects(completion)` : LLM이 JSON 앞뒤에 다른 말을 섞어 변환에 실패하면, 아래 보조 함수로 JSON 부분만 찾아냄
- `if "domain" in completion.keys():` : 결과에 `domain` 이 있으면
  - `is_finance = domain == "finance"` : `==` 비교의 결과(True/False)를 변수에 저장
- `return completion, is_finance` : (추출 결과, 금융 여부) 두 값 반환

**`extract_json_objects(text, decoder=JSONDecoder())`** — 글 속에서 JSON 찾기
- `text.find("{", pos)` : `pos` 위치부터 `{` 가 처음 나오는 곳. 없으면 `-1`
- `decoder.raw_decode(text[match:])` : 그 위치부터 JSON으로 해석해 보고, (결과, 끝난 위치)를 돌려줌
- 성공하면 결과를 모으고 그 다음부터 계속 탐색, 실패(`ValueError`)하면 한 글자 뒤부터 재시도
- `while True:` + `break` : 더 이상 `{` 가 없을 때까지 무한 반복

> 🔰 함수는 **호출되는 시점**에만 존재하면 되므로, `generate_query` 가 아래쪽에 정의된 `extract_json_objects` 를 사용해도 문제없습니다.

In [ ]:
# 8
### YOUR CODE HERE ###

import json
from openai import OpenAI
from json import JSONDecoder

oai_client = OpenAI()

def generate_query(query):
    llm_input = prompt_generator(query)
    completion = oai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=
    llm_input
    ).choices[0].message.content

    try:
        completion = json.loads(completion)
    except:
        completion = extract_json_objects(completion)

    if "domain" in completion.keys():
        domain = completion["domain"]
        is_finance = domain == "finance"
    else:
        is_finance = False

    return completion, is_finance

def extract_json_objects(text, decoder=JSONDecoder()):
    """Find JSON objects in text, and yield the decoded JSON data
    """
    pos = 0
    results = []
    while True:
        match = text.find("{", pos)
        if match == -1:
            break
        try:
            result, index = decoder.raw_decode(text[match:])
            results.append(result)
            pos = match + index
        except ValueError:
            pos = match + 1
    return results

Let’s verify whether the above code works correctly in practice.

### 💡 [# 9] 코드 설명 — 금융 질문 3개로 시험

**한 줄 요약:** 14, 53, 64번 데이터의 질문을 구조화해 결과를 확인합니다.

- `dataset_path = "/path/to/..."` : ⚠️ **실제 데이터 파일 경로로 수정**
- `if repeat >= 65: break` : 65번 이후는 보지 않음
- `if repeat not in [14, 53, 64]: repeat += 1; continue` : 원하는 번호가 아니면 건너뜀
- `generate_query(item['query'])` : 결과는 `(추출된 사전, True/False)` 형태로 출력됩니다.

In [ ]:
# 9
### YOUR CODE HERE ###
import bz2
import json

dataset_path = "/path/to/CRAG dataset/crag_task_1_dev_v4_release.jsonl.bz2"
repeat = 0
with bz2.open(dataset_path, "rt") as file:
    for line in file:
        if repeat >= 65:
          break
        if repeat not in [14, 53, 64]:
          repeat += 1
          continue
        item = json.loads(line)
        print(f"query: {item['query']}")
        generated_query = generate_query(item['query'])
        print(f"generated_query: {generated_query}")
        repeat += 1

# print(f"generated_query: {generated_query}")

### 4. Implementing a Query Executor

We are now at the most challenging step. Even if we generate a query to send to the API, we cannot directly pass it as is.

Therefore, it is necessary to transform the generated results into a format where only the required elements are sent to the API.

Additionally, simply being able to call the API does not solve the problem. Many of the data points in the CRAG dataset require multiple pieces of information. As a result, a single API call may not provide sufficient information to generate an answer.

To address this, we need to predefine a **decision tree** outlining how we will use and analyze the API. With this decision tree set, the necessary code can then be written.

For now, let’s declare some **utility functions** that may be needed later. These functions will transform sentences into a fixed format using methods like pattern matching or other techniques.

### 💡 [# 10] 코드 설명 — 이름·날짜를 다듬는 보조 함수들

**한 줄 요약:** API 응답에서 값을 찾기 쉽게 이름과 날짜 형식을 통일해 주는 도우미 함수 5개입니다.

- `normalize_key(key)` : `re.sub(r'[^a-zA-Z0-9]', '', key).lower()` — **정규표현식**으로 영문자·숫자가 아닌 것을 모두 지우고 소문자로. 예: `"P/E Ratio"` → `"peratio"`
- `get_metric_from_response(response, metric)` : 응답 사전에서, 다듬은 이름이 `metric` 과 같은 항목의 값을 찾아 반환. 없으면 `None`
- `convert_to_standard_format(date_string)` : 여러 형태의 날짜 글자를 KG가 쓰는 표준 형식으로 변환
  - `parser.parse(date_string)` : "2024-03-01", "March 1, 2024" 등 다양한 표기를 날짜 객체로 해석
  - `pytz.timezone('US/Eastern')` : 미국 동부 시간대(미국 주식시장 기준)
  - `dt.tzinfo is None` : 시간대 정보가 없으면 동부 시간으로 지정(`localize`), 있으면 동부 시간으로 환산(`astimezone`)
  - `dt.replace(hour=0, minute=0, second=0, microsecond=0)` : 시각을 자정(00:00:00)으로 맞춤
  - `dt.strftime('%Y-%m-%d %H:%M:%S %Z')` : `2024-03-01 00:00:00 EST` 같은 글자로 변환
  - 해석 실패 시 원래 글자를 그대로 반환
- `add_one_day` / `subtract_one_day` : 같은 방식으로 **하루 뒤 / 하루 전** 날짜 글자를 만듦 (`timedelta(days=1)` 은 "하루"라는 시간 간격)

> 🔰 하루 전·후가 필요한 이유: 주말·휴일에는 주식 데이터가 없어서 딱 그 날짜가 없을 때 인접한 날을 대신 찾아보기 위해서입니다.

In [ ]:
# 10
### YOUR CODE HERE ###

import datetime
from datetime import timedelta
from dateutil import parser
import pytz
import re

def normalize_key(key):
    return re.sub(r'[^a-zA-Z0-9]', '', key).lower()

def get_metric_from_response(response, metric):
    normalized_metric = normalize_key(metric)
    if response != None:
        for key, value in response.items():
            if normalize_key(key) == normalized_metric:
                return value
    return None


def convert_to_standard_format(date_string):
    try:
        dt = parser.parse(date_string)

        est = pytz.timezone('US/Eastern')

        if dt.tzinfo is None:
            dt = est.localize(dt)
        else:
            dt = dt.astimezone(est)
        dt = dt.replace(hour=0, minute=0, second=0, microsecond=0)

        formatted_date = dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        return formatted_date
    except (ValueError, OverflowError) as e:
        return date_string

def add_one_day(date_string):
    try:
        dt = parser.parse(date_string)

        est = pytz.timezone('US/Eastern')

        if dt.tzinfo is None:
            dt = est.localize(dt)
        else:
            dt = dt.astimezone(est)

        dt_plus_one = dt + timedelta(days=1)
        dt = dt.replace(hour=0, minute=0, second=0, microsecond=0)
        formatted_date = dt_plus_one.strftime('%Y-%m-%d %H:%M:%S %Z')
        return formatted_date
    except (ValueError, OverflowError) as e:
        return f"Invalid date string: {e}"

def subtract_one_day(date_string):
    try:
        dt = parser.parse(date_string)

        est = pytz.timezone('US/Eastern')

        if dt.tzinfo is None:
            dt = est.localize(dt)
        else:
            dt = dt.astimezone(est)

        dt_minus_one = dt - timedelta(days=1)
        dt = dt.replace(hour=0, minute=0, second=0, microsecond=0)
        formatted_date = dt_minus_one.strftime('%Y-%m-%d %H:%M:%S %Z')
        return formatted_date
    except (ValueError, OverflowError) as e:
        return f"Invalid date string: {e}"


Now, let’s write a function that follows the **decision tree** we established to interact with the API and process its results. A decision tree is a decision support structure that uses a tree-like model of decisions and their possible consequences, including outcomes, costs, and utility. We'll be using our pre-built API in this manner.

<img src="https://i.imgur.com/F7lfJQq.png">

Our search process will proceed as follows:  

1.	Extract relevant entities from the question that are required to use the API.
2.	Based on the extracted entities, determine the necessary parameters to call finance-related APIs.
3.	Use the identified parameters to call the relevant API and retrieve the results.
4.	Filter the results to extract only the relevant information.

By following these steps, we will ultimately obtain the kg_results, which contain the relevant information extracted from the Knowledge Graph (KG).


### 💡 [# 11] 코드 설명 — 구조화된 질문으로 KG 조회하기 (Query Executor)

**한 줄 요약:** # 8에서 뽑은 (회사, 지표, 날짜)에 맞는 API를 호출하고, 해당 날짜의 데이터만 골라 글로 정리합니다. 이 노트북에서 가장 긴 함수이지만 **4단계**로 나누면 쉽습니다.

**① 날짜 정리**
- `generated_query['datetime'].split(' - ')` : "시작 - 끝" 형태면 둘로 나눔
- 각 날짜를 `convert_to_standard_format` 으로 표준화해 `formatted_time_list` 에 저장

**② 회사 목록 만들기**
- `market_identifier` 가 글자면 `split(",")` 로 쉼표 기준 분리, 이미 리스트면 그대로 사용 (`isinstance(값, str)` = "글자인가?")

**③ 회사마다 API 호출** (`for company_name in company_names:`)
- `api.finance_get_company_name(company_name)["result"]` : 정식 회사명 찾기
- 결과가 비어 있으면(`res == []`) 입력값을 대문자로 바꿔 티커로 간주, 아니면 `finance_get_ticker_by_name` 으로 티커 조회
- `generated_query['metric'].lower().strip()` 값에 따라 `if/elif` 로 알맞은 API 호출
- 어디에도 해당하지 않으면 `finance_get_info` 로 종합 정보를 받아 `get_metric_from_response` 로 원하는 항목만 추출

**④ 날짜로 걸러내기**
- 응답에 그 날짜가 있으면 그 값, 없으면 하루 뒤, 그것도 없으면 하루 전, 모두 없으면 전체를 사용
- `copy.deepcopy(...)` : 원본을 건드리지 않도록 **완전한 복사본** 생성
- `kg_results.append({ "회사 지표": 값, 'time': 날짜 })` : 결과 누적
- 안쪽 `try/except` : 날짜 거르기에 실패하면 응답 전체를 넣음
- 바깥 `try/except Exception as e` : API 호출 자체가 실패하면 메시지만 찍고 넘어감 (`pass` = 아무것도 안 함)

**마무리**
- `"<DOC>\n".join([str(res) for res in kg_results]) if len(kg_results) > 0 else ""` : 결과가 있으면 각 항목을 글자로 바꿔 `<DOC>` 구분자로 이어 붙이고, 없으면 빈 문자열. (`A if 조건 else B` = 한 줄 조건문)

> 🔰 이 함수는 # 5에서 만든 전역 변수 `api` 를 사용합니다. 그래서 # 5를 먼저 실행해야 합니다.

In [ ]:
# 11
### YOUR CODE HERE ###

import copy

def get_finance_kg_results(generated_query):
    formatted_time_list = []
    if 'datetime' in generated_query:
        datetime_list = generated_query['datetime'].split(' - ')
        for datetime in datetime_list:
            formatted_time_list.append(convert_to_standard_format(datetime.strip()))


    kg_results = []
    res = ""
    if "market_identifier" in generated_query.keys() and generated_query["market_identifier"] is not None:
        if isinstance(generated_query["market_identifier"], str):
            company_names = generated_query["market_identifier"].split(",")
        else:
            company_names = generated_query["market_identifier"]

        for company_name in company_names:
            try:
                res = api.finance_get_company_name(company_name)["result"]

                if res == []:
                    ticker_name = company_name.upper()
                else:
                    ticker_name = api.finance_get_ticker_by_name(res[0])["result"]

                if generated_query['metric'].lower().strip() == 'price':
                    response = api.finance_get_price_history(ticker_name)['result']
                elif generated_query['metric'].lower().strip() == 'dividend':
                    response = api.finance_get_dividends_history(ticker_name)['result']
                elif generated_query['metric'].lower().strip() == 'p/e ratio':
                    response = api.finance_get_pe_ratio(ticker_name)['result']
                elif generated_query['metric'].lower().strip() == 'eps':
                    response = api.finance_get_eps(ticker_name)["result"]
                elif generated_query['metric'].lower().strip() == 'marketcap' :
                    response = api.finance_get_market_capitalization(ticker_name)['result']
                else:
                    response = api.finance_get_info(ticker_name)['result']
                    metric_value = get_metric_from_response(response, generated_query['metric'])
                    if metric_value is not None:
                        response = metric_value

                try:
                    for formatted_time in formatted_time_list:
                        if formatted_time in response:
                            filtered_response = copy.deepcopy(response[formatted_time])
                        elif add_one_day(formatted_time) in response:
                            filtered_response = copy.deepcopy(response[add_one_day(formatted_time)])
                        elif subtract_one_day(formatted_time) in response:
                            filtered_response = copy.deepcopy(response[subtract_one_day(formatted_time)])
                        else:
                            filtered_response = copy.deepcopy(response)
                        kg_results.append({company_name + " " + generated_query["metric"]: filtered_response, 'time': formatted_time})
                except:
                    kg_results.append({company_name + " " + generated_query["metric"]: response})

            except Exception as e:
                print("Fail to parse the generated query")
                pass

    kg_results = "<DOC>\n".join([str(res) for res in kg_results]) if len(kg_results) > 0 else ""
    return  kg_results


Let’s now verify whether we can obtain the kg_results using real data by executing the pipeline we just declared.  

This will confirm that our process, from entity extraction to API interaction, works correctly.

### 💡 [# 12] 코드 설명 — 실제 데이터로 KG 조회 시험

**한 줄 요약:** 데이터에서 금융 질문을 찾아 KG 결과가 처음으로 얻어질 때까지 시도합니다.

- `if item["domain"] == "finance":` : 금융 분야 데이터만
- `generated_query, is_finance = generate_query(item['query'])` : 반환된 두 값을 각각 받음
- `kg_results = get_finance_kg_results(generated_query)` : KG 조회
- `if kg_results not in ["", None]: break` : 결과가 비어 있지 않으면 성공이므로 종료

In [ ]:
# 12
### YOUR CODE HERE ###

import bz2
import json

with bz2.open(dataset_path, "rt") as file:
    for line in file:
        item = json.loads(line)
        print(f"query: {item['query']}")

        if item["domain"] == "finance":
          generated_query, is_finance = generate_query(item['query'])
          if is_finance:
            print("generated_query: ", generated_query)
            kg_results = get_finance_kg_results(generated_query)
            if kg_results not in ["", None]:
                break

print(f"kg_results: {kg_results}")

###5. Implementing a Mock KG Query Engine

Let’s now define the `KGQueryEngine` that brings together all the components we have implemented.

When a query is provided, the `KGQueryEngine` will interact with the Knowledge Graph (KG) to retrieve the relevant information.

### 💡 [# 13] 코드 설명 — `KGQueryEngine` 클래스로 묶기

**한 줄 요약:** 지금까지 만든 "질문 분석(# 8) + KG 조회(# 11)"를 하나의 클래스로 정리합니다.

- **`query(self, query)`** : 전체 흐름
  - `self.generate_query(query)` : 질문 → 구조화된 JSON + 금융 여부
  - 금융이면 `self.get_finance_kg_results(...)` 로 조회, 아니면 빈 문자열
  - `return kg_results, is_finance`
- **`generate_query`** : # 8과 같은 내용
- **`get_finance_kg_results`** : # 11과 같은 내용
- **`prompt_generator`** : # 7과 같은 내용

> 🔰 새로운 로직은 없습니다. 흩어진 함수들을 클래스 안으로 옮긴 것이며, 함수 정의에 `self` 가 추가되고 호출할 때 `self.` 을 붙이는 점만 다릅니다. 이렇게 묶어 두면 `KGQueryEngine().query("질문")` 한 줄로 쓸 수 있습니다.

In [ ]:
# 13
### YOUR CODE HERE ###

class KGQueryEngine:
    def query(self, query):
        generated_query, is_finance = self.generate_query(query)

        if is_finance:
            kg_results = self.get_finance_kg_results(generated_query)
        else:
            kg_results = ""

        return kg_results, is_finance

    def generate_query(self, query):
        llm_input = prompt_generator(query)
        completion = oai_client.chat.completions.create(
        model="gpt-3.5-turbo",
        temperature=0,
        messages=
        llm_input
        ).choices[0].message.content

        try:
            completion = json.loads(completion)
        except:
            completion = extract_json_objects(completion)

        if "domain" in completion.keys():
            domain = completion["domain"]
            is_finance = domain == "finance"
        else:
            is_finance = False

        return completion, is_finance

    def get_finance_kg_results(self, generated_query):
        formatted_time_list = []
        if 'datetime' in generated_query:
            datetime_list = generated_query['datetime'].split(' - ')
            for datetime in datetime_list:
                formatted_time_list.append(convert_to_standard_format(datetime.strip()))


        kg_results = []
        res = ""
        if "market_identifier" in generated_query.keys() and generated_query["market_identifier"] is not None:
            if isinstance(generated_query["market_identifier"], str):
                company_names = generated_query["market_identifier"].split(",")
            else:
                company_names = generated_query["market_identifier"]

            for company_name in company_names:
                try:
                    res = api.finance_get_company_name(company_name)["result"]

                    if res == []:
                        ticker_name = company_name.upper()
                    else:
                        ticker_name = api.finance_get_ticker_by_name(res[0])["result"]

                    if generated_query['metric'].lower().strip() == 'price':
                        response = api.finance_get_price_history(ticker_name)['result']
                    elif generated_query['metric'].lower().strip() == 'dividend':
                        response = api.finance_get_dividends_history(ticker_name)['result']
                    elif generated_query['metric'].lower().strip() == 'p/e ratio':
                        response = api.finance_get_pe_ratio(ticker_name)['result']
                    elif generated_query['metric'].lower().strip() == 'eps':
                        response = api.finance_get_eps(ticker_name)["result"]
                    elif generated_query['metric'].lower().strip() == 'marketcap' :
                        response = api.finance_get_market_capitalization(ticker_name)['result']
                    else:
                        response = api.finance_get_info(ticker_name)['result']
                        metric_value = get_metric_from_response(response, generated_query['metric'])
                        if metric_value is not None:
                            response = metric_value

                    try:
                        for formatted_time in formatted_time_list:
                            if formatted_time in response:
                                filtered_response = copy.deepcopy(response[formatted_time])
                            elif add_one_day(formatted_time) in response:
                                filtered_response = copy.deepcopy(response[add_one_day(formatted_time)])
                            elif subtract_one_day(formatted_time) in response:
                                filtered_response = copy.deepcopy(response[subtract_one_day(formatted_time)])
                            else:
                                filtered_response = copy.deepcopy(response)
                            kg_results.append({company_name + " " + generated_query["metric"]: filtered_response, 'time': formatted_time})
                    except:
                        kg_results.append({company_name + " " + generated_query["metric"]: response})

                except Exception as e:
                    print("Fail to parse the generated query")
                    pass

        kg_results = "<DOC>\n".join([str(res) for res in kg_results]) if len(kg_results) > 0 else ""
        return  kg_results

    def prompt_generator(self, query):
        user_message = ""
        user_message += f"Query: {query}\n"

        llm_input = [
          {"role": "system", "content": entity_extract_template},
          {"role": "user", "content": user_message},
        ]

        return llm_input

## II. Implementing a Reader

Before defining RAG with KG, let’s finalize the `Reader`. This `Reader` will be the same as the one you used in Task 1.

Here’s a reminder of the structure:

### 💡 [# 14] 코드 설명 — 답변기(Reader) 클래스

**한 줄 요약:** 참고자료와 질문을 받아 LLM으로 짧은 답을 만드는 클래스입니다(Task 1과 동일).

- `self.system_prompt` : "최대한 짧게 답하고, 참고자료에 없으면 'I don't know'라고 하라"
- `generate_response(self, question, top_k_chunks)` : 프롬프트 조립 → LLM 호출(`temperature=0`) → 답 반환
- `prompt_generator(self, query, top_k_chunks)` : 참고자료를 `- 내용` 목록으로 만들고 4000자로 자른 뒤 질문과 합쳐 메시지 구성

In [ ]:
# 14
### YOUR CODE HERE ###

from openai import OpenAI

oai_client = OpenAI()

class Reader:
  def __init__(self):

    self.system_prompt = """
    You are provided with a question and various references.
    Your task is to answer the question succinctly, using the fewest words possible.
    If the references do not contain the necessary information to answer the question, respond with 'I don't know'.
    There is no need to explain the reasoning behind your answers.
    """

  def generate_response(self, question: str, top_k_chunks: list) -> str:
      """
      Generate answer from context.
      """
      llm_input = self.prompt_generator(question, top_k_chunks)
      completion = oai_client.chat.completions.create(
      model="gpt-3.5-turbo",
      temperature=0,
      messages=
      llm_input
      ).choices[0].message.content
      return completion

  def prompt_generator(self, query, top_k_chunks):
      user_message = ""
      references = ""

      if len(top_k_chunks) > 0:
          references += "# References \n"
          # Format the top sentences as references in the model's prompt template.
          for chunk_id, chunk in enumerate(top_k_chunks):
              references += f"- {chunk.strip()}\n"

      references = references[:MAX_CONTEXT_REFERENCES_LENGTH]
      # Limit the length of references to fit the model's input size.

      user_message += f"{references}\n------\n\n"
      user_message
      user_message += f"Using only the references listed above, answer the following question: \n"
      user_message += f"Question: {query}\n"

      llm_input = [
        {"role": "system", "content": self.system_prompt},
        {"role": "user", "content": user_message},
      ]

      return llm_input

Let’s verify once again that the model is functioning correctly.

Run the following test to ensure all components are working as expected.


### 💡 [# 15] 코드 설명 — Reader 동작 확인

**한 줄 요약:** 참고자료 없이(`[]`) 첫 질문을 Reader에 넣어 정상 동작하는지 봅니다.

- 참고자료가 없으므로 "I don't know"가 나오면 정상입니다.

In [ ]:
# 15
### YOUR CODE HERE ###

reader = Reader()
import bz2
import json

with bz2.open(dataset_path, "rt") as file:
    for line in file:
        item = json.loads(line)
        print(f"query: {item['query']}")
        print()
        answer = reader.generate_response(item['query'], [])
        break

print(f"answer: {answer}")

## III. Implementing an LLM + Mock KG

Finally, let’s define our RAG system, which is the ultimate goal of this session. Fortunately, we have already completed the complex steps above. To define the RAG system, we simply need to combine the previously defined components into a single class.  


### 💡 [# 16] 코드 설명 — LLM + KG RAG 클래스

**한 줄 요약:** KG 조회 엔진과 Reader를 합친 `RAGWithKG` 를 정의합니다.

- `__init__` : `KGQueryEngine()` 과 `Reader()` 를 부품으로 보관
- `inference(self, query)` :
  1. `self.kg_query_engine.query(query)` : KG에서 관련 데이터 조회
  2. `self.reader.generate_response(query, [kg_results])` : KG 결과를 **리스트로 감싸** 참고자료로 전달 (Reader가 리스트를 기대하기 때문)
  3. `return answer, kg_results`

In [ ]:
# 16
### YOUR CODE HERE ###

class RAGWithKG:
    def __init__(self):
        self.kg_query_engine = KGQueryEngine()
        self.reader = Reader()

    def inference(self, query):
        # 1. retrieve relevant kg results
        kg_results, is_finance = self.kg_query_engine.query(query)

        # 2. answer the question based on the retrieved chunks
        answer = self.reader.generate_response(query, [kg_results])

        return answer, kg_results

Let’s now verify whether the implemented RAG system operates as intended. This system retrieves relevant information from the Knowledge Graph (KG) and utilizes it to generate a final answer.

Additionally, we will check whether the system can correctly handle the following queries, which the RAG system from Task 1 previously failed to answer.

<br/>  
Question: **What is the ex-dividend date of microsoft in the 1st qtr of 2024**.   
Answer: **The ex-dividend date of microsoft in the 1st qtr of 2024 is feb 14, 2024**
<br/>

<br/>  
Question: **I'm looking for the p/e ratio of dks. would you happen to know what it is?**.   
Answer: **13.75**
<br/>

<br/>  
Question: **What's auph's earnings per share?**.   
Answer: **0.4**
<br/>


### 💡 [# 17] 코드 설명 — 금융 질문 3개로 LLM + KG 시험

**한 줄 요약:** Task 1에서 틀렸던 14, 53, 64번 금융 질문을 KG 기반 RAG로 다시 풉니다.

- `rag = RAGWithKG()` : 객체 생성
- 번호 필터링은 # 9와 같은 방식
- 예측 답, 정답, 그리고 근거가 된 `kg results` 를 출력

> 🔰 웹 검색만 쓰던 Task 1의 결과와 비교해 보세요.

In [ ]:
# 17
### YOUR CODE HERE ###

rag = RAGWithKG()
repeat = 0

with bz2.open(dataset_path, "rt") as file:
    for line in file:
        item = json.loads(line)
        if repeat >= 65:
          break
        if repeat not in [14, 53, 64]:
            repeat += 1
            continue

        print(f"query: {item['query']}")
        print()
        answer, kg_results = rag.inference(item['query'])
        print(f"predicted answer: {answer}")
        print(f"ground truth answer: {item['answer']}")
        print()
        print(f"kg results: {kg_results}")

        repeat += 1

## IV. Implementing an LLM + Web Search Results + Mock KG

### 💡 [# 18] 코드 설명 — 웹 검색용 검색기 다시 정의

**한 줄 요약:** KG와 웹 검색을 함께 쓰기 위해, Task 1의 `parse_htmls` 와 `LlamaIndexRetriever` 를 이 노트북에도 정의합니다.

- `Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")` : 임베딩 모델 지정
- `parse_htmls(search_results)` : HTML → 글
- `LlamaIndexRetriever.retrieve(query, search_results, topk)` : 글을 `Document` 로 감싸기 → 512토큰 조각으로 인덱스 생성 → 상위 k개 검색 → 글만 리스트로 반환

> 🔰 내용은 Task 1의 # 4, # 8과 동일합니다.

In [ ]:
# 18
### YOUR CODE HERE ###

from llama_index.core.schema import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

def parse_htmls(search_results):
    all_documents = []

    # Process each HTML text from the search results to extract text content.
    for html_text in search_results:

        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(html_text["page_result"], features="lxml")
        text = soup.get_text(" ", strip=True)  # Use space as a separator, strip whitespaces
        all_documents.append(text)

    return all_documents

class LlamaIndexRetriever:
  def __init__(self):
      self.parser = SentenceSplitter(chunk_size=512, chunk_overlap=0)

  def retrieve(self, query, search_results, topk):
      documents = []

      for document in parse_htmls(search_results):
        if not document:
            # If no text is extracted, add an empty string as a placeholder.
            documents.append(Document(text=""))
        else:
            documents.append(Document(text=document))

      # Split documents into chunks & Create vector index
      base_index = VectorStoreIndex.from_documents(documents = documents, transformations=[self.parser])

      # Execute query
      base_retriever = base_index.as_retriever(similarity_top_k=topk)

      retrieved_nodes = base_retriever.retrieve(query)

      retrieved_results = [retrieved_node.node.get_content().strip() for retrieved_node in retrieved_nodes]

      return retrieved_results

### 💡 [# 19] 코드 설명 — 웹 검색 + KG를 모두 쓰는 RAG

**한 줄 요약:** 질문이 금융이면 KG 결과를, 아니면 웹 검색 결과를 참고자료로 쓰는 `RAGWithSRKG` 를 정의합니다.

- `__init__` : 검색기, KG 엔진, Reader 세 부품 보관
- `inference(self, query, search_results, topk)` :
  1. 웹 검색 결과에서 관련 조각 검색
  2. KG 조회 (+ 금융 여부 확인)
  3. `if is_finance: combined_results = [kg_results]` / `else: combined_results = retrieved_results` : **분야에 따라 참고자료를 선택**(이런 방식을 "라우팅"이라고 합니다)
  4. Reader로 답 생성
- 주석 처리된 두 줄은 "KG 결과와 웹 결과를 **둘 다** 넣는" 다른 방법입니다. `#` 를 풀어 비교 실험해 볼 수 있습니다.

In [ ]:
# 19
### YOUR CODE HERE ###

class RAGWithSRKG:
    def __init__(self):
        self.retriever = LlamaIndexRetriever()
        self.kg_query_engine = KGQueryEngine()
        self.reader = Reader()

    def inference(self, query, search_results, topk):
        # 1. retrieve relevant chunks
        retrieved_results = self.retriever.retrieve(query, search_results, topk)

        # 2. retrieve relevant kg results
        kg_results, is_finance = self.kg_query_engine.query(query)

        # combined_results = [kg_results]
        # combined_results.extend(retrieved_results)
        if is_finance:
          combined_results = [kg_results]
        else:
          combined_results = retrieved_results

        # 3. answer the question based on the retrieved chunks
        answer = self.reader.generate_response(query, combined_results)

        return answer, combined_results

Let us evaluate whether it can successfully answer all the following queries.

<br/>  
Question: **In 2004, which animated film was recognized with the best animated feature film oscar?**.   
Answer: **Finding Nemo**
<br/>

<br/>  
Question: **What is the ex-dividend date of microsoft in the 1st qtr of 2024**.   
Answer: **The ex-dividend date of microsoft in the 1st qtr of 2024 is feb 14, 2024**
<br/>

<br/>  
Question: **I'm looking for the p/e ratio of dks. would you happen to know what it is?**.   
Answer: **13.75**
<br/>

<br/>  
Question: **What's auph's earnings per share?**.   
Answer: **0.4**
<br/>

### 💡 [# 20] 코드 설명 — 최종 RAG 시험

**한 줄 요약:** 일반 질문(5번)과 금융 질문(14, 53, 64번)을 모두 최종 RAG로 풀어 봅니다.

- `if repeat not in [5, 14, 53, 64]:` : 네 개 번호만 처리
- 답, 정답, 사용된 참고자료를 출력

> 🔰 5번은 웹 검색 결과로, 나머지는 KG 결과로 답하게 됩니다. 각 질문에서 "retrieved results"가 어떤 종류인지 확인해 보세요.

In [ ]:
# 20
### YOUR CODE HERE ###

rag = RAGWithSRKG()
topk = 5

repeat = 0
with bz2.open(dataset_path, "rt") as file:
    for line in file:
        if repeat >= 65:
          break
        if repeat not in [5, 14, 53, 64]:
            repeat += 1
            continue

        item = json.loads(line)
        print(f"query: {item['query']}")
        print()
        answer, retrieved_results = rag.inference(item['query'], item['search_results'], topk)
        print(f"predicted answer: {answer}")
        print(f"ground truth answer: {item['answer']}")
        print()
        print("retrieved results:")
        for rank, retrieved_result in enumerate(retrieved_results):
            print(f"{rank}: {retrieved_result}")
        print()
        repeat += 1